## Case Study - Household response to inflation 

## Imports and global style

In [ ]:
import pandas as pd
import numpy as np 
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns 
from scipy import stats

%matplotlib inline 

plt.rcParams['font.size'] = 11
plt.rcParams['axes.titlesize'] = 12
plt.rcParams['axes.labelsize'] = 11
plt.rcParams['xtick.labelsize'] = 10
plt.rcParams['ytick.labelsize'] = 10
plt.rcParams['font.family'] = 'sans serif'

print('Imports - successful!')

## Data Import

In [ ]:
df = pd.read_csv('topic6_147200_147650.csv')
print('Data import - successful\n')
print('Shape:', df.shape)
print('\nColumns:', df.columns.tolist())
print()
df.head(int(30))



## Global variable encodings: 
Due to the fact that most of the data is in ordinal scale (e.g. "Prices want up a lot") we have to assaign ranks to the data in order to use Spearman rank correlation. We use maps to encode text answers from a survey as numbers (1-5).

In [156]:
price_past_map={
    'Prices went down a lot': 1,
    'Prices went down little': 2,
    'Prices stayed exactly the same': 3,
    'Prices went up a little': 4,
    'Prices went up a lot': 5
}
price_exp_map={
    'Prices will decrease a lot': 1,
    'Prices will decrease a little': 2,
    'Prices will be exactly the same': 3,
    'Prices will increase a little': 4,
    'Prices will increase a lot': 5
}
sit_map={
    'Much worse off':1,
    'Somewhat worse off':2,
    'About the same':3,
    'Somewhat better off':4,
    'Much better off':5
}
income_map={
    'First quintile':1,
    'Second quintile':2,
    'Third quintile':3,
    'Fourth quintile':4,
    'Fifth quintile':5
}
age_map={
    '18-34':1, 
    '35-49':2,
    '50-70':3,
    '70 and older':4
}

df['price_past_num'] = df['price_past'].map(price_past_map)
df['price_exp_num'] = df['price_exp'].map(price_exp_map)
df['sit_last_num'] = df['sit_last_12months'].map(sit_map)
df['sit_next_num'] = df['sit_next_12months'].map(sit_map)
df['income_num'] = df['income'].map(income_map)
df['age_num'] = df['age_group'].map(age_map)

df['reduced'] = (df['reduced_consumption'] == 'Yes (cheaper or fewer goods)').astype(int)
df['increased_income'] = (df['increased_income'] == 'Yes (raise or second job)').astype(int)
df['increased_spending'] = (df['increased_spending'] == 'Yes (purchased durable goods)').astype(int)
df['n_strategies'] = df['reduced'] + df['increased_income'] + df['increased_spending']

PAST_ORDER = list(price_past_map.keys())
EXP_ORDER = list(price_exp_map.keys())
INCOME_ORDER = list(income_map.keys())
INCOME_SHORT = ['Q1\n(Lowest)', 'Q2', 'Q3', 'Q4', 'Q5\n(Highest)']
AGE_ORDER = list(age_map.keys())
PAST_SHORT = ['Down a lot', 'Down little', 'Same', 'Up a little', 'Up a lot']
EXP_SHORT = ['Down a lot', 'Down a little', 'Same', 'Increase a little', 'Increase a lot']

print('Setup test - successful' )
#print("reduced:", df['reduced'].value_counts())
#print("price expectations:", df['price_exp'].value_counts())
#print("increased_income:", df['increased_income'].value_counts())
#print("increased_spending:", df['increased_spending'].value_counts())

Setup test - successful


## Figure 1 - Distributions perceptions and expectations 
 

In [ ]:
#print(df['price_past'].value_counts())
past_percent = [df['price_past'].value_counts().get(cat, 0) / len(df) * 100 for cat in PAST_ORDER]
exp_percent = [df['price_exp'].value_counts().get(cat, 0) / len(df) * 100 for cat in EXP_ORDER]

fig,axes = plt.subplots(1,2, figsize=(13,5))

bar1 = axes[0].bar(PAST_SHORT, past_percent,color='red',linewidth=0.8)

axes[0].bar_label(bar1, fmt='%.2f%%', padding=3, fontsize=12, fontweight='bold')

axes[0].set_title('Price perceptions\n(past 12 months)', fontweight='bold')
axes[0].set_ylabel('% of respondents')
axes[0].set_ylim(0, 85)
axes[0].spines['top'].set_visible(False)
axes[0].spines['right'].set_visible(False)
axes[0].set_xticks(range(len(PAST_ORDER)))
axes[0].set_xticklabels(PAST_ORDER, rotation=25, ha='right')


bar2 = axes[1].bar(EXP_SHORT, exp_percent,color='blue', linewidth=0.8)
axes[1].bar_label(bar2, fmt='%.2f%%', padding=3, fontsize=12, fontweight='bold')
axes[1].set_title('Price expectations\n(next 12 months)', fontweight='bold')
axes[1].set_ylabel('% of respondents')
axes[1].set_ylim(0, 85)
axes[1].spines['top'].set_visible(False)
axes[1].spines['right'].set_visible(False)

plt.suptitle('Figure 1: Distribution of inflation perceptions and expectations',fontsize=13, y=1.02)
plt.tight_layout()
plt.show()


## Figure 2 - Household coping strategies

**What we do:** show adoption rates for each strategy, and how many households used more than one simultaneously.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

#Panel A: adoption rate of each strategy 
strategy_labels = ['Reduce\nconsumption', 'Increase\nincome', 'Increase\nspending']
strategy_percentages   = [df['reduced'].mean() * 100, df['increased_income'].mean() * 100, df['increased_spending'].mean() * 100]
strategy_colors = ['#e74c3c', '#3498db', '#f39c12']

bar_a = axes[0].bar(strategy_labels, strategy_percentages,
                     color=strategy_colors, edgecolor='white')

for bar, val in zip(bar_a, strategy_percentages):
    axes[0].text(bar.get_x() + bar.get_width() / 2,
                 val + 0.5, f'{val:.2f}%',
                 ha='center', fontsize=12, fontweight='bold')

axes[0].set_ylim(0, 75)
axes[0].set_ylabel('% of respondents')
axes[0].set_title('A. Strategy adoption rate', fontweight='bold')
axes[0].spines['top'].set_visible(False)
axes[0].spines['right'].set_visible(False)

#Panel B: number of simultaneous strategies 
#value_counts() counts how many respondents used 0, 1, 2, or 3 strategies
n_counts = df['n_strategies'].value_counts().reindex([0, 1, 2, 3], fill_value=0)
n_percentages   = n_counts / len(df) * 100
n_labels = ['None', 'One', 'Two', 'All three']
n_colors = ['#95a5a6', '#3498db', '#e74c3c', '#8e44ad']

bar_b = axes[1].bar(n_labels, n_percentages, color=n_colors, edgecolor='white')

for bar, val in zip(bar_b, n_percentages):
    axes[1].text(bar.get_x() + bar.get_width() / 2,
                 val + 0.3, f'{val:.2f}%',
                 ha='center', fontsize=11, fontweight='bold')

axes[1].set_ylabel('% of respondents')
axes[1].set_title('B. Number of simultaneous strategies', fontweight='bold')
axes[1].spines['top'].set_visible(False)
axes[1].spines['right'].set_visible(False)

plt.suptitle('Figure 2: Household coping strategies', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

## Figure 3 — Income vs perceptions/expectations 
**What we do:** For each income group we show whether there is relation (positive/negative/no relation) between income and perceptions and expectations using Spearman correlation coefficient. 

In [ ]:
#First we have to compute the statistical tests
r_past, p_past = stats.spearmanr(df['income_num'], df['price_past_num'])
r_exp,  p_exp  = stats.spearmanr(df['income_num'], df['price_exp_num'])

print(f'Income vs past perception:    Spearman r = {r_past:.3f}, p = {p_past:.4f}')
print(f'Income vs future expectation: Spearman r = {r_exp:.3f},  p = {p_exp:.4f}')